# 08. 검색 품질 분석 (Retrieval Analysis)

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. VectorStoreRetriever의 search_type별(similarity, mmr) Trace 차이를 LangSmith에서 비교할 수 있어요
2. EnsembleRetriever(BM25 + FAISS)의 병렬 실행 구조를 Trace 계층도로 이해할 수 있어요
3. `metadata` 태깅으로 검색 설정 A/B 비교 실험을 체계적으로 관리할 수 있어요
4. ContextualCompressionRetriever를 사용한 Reranker 적용 전후의 검색 품질 변화를 분석할 수 있어요
5. LangSmith SDK로 Retriever run을 프로그래밍 방식으로 조회하고 분석할 수 있어요

## 사전 지식

- 07-Document-Loading-Tracing 완료 (VectorStore, Retriever 추적 기본)
- EnsembleRetriever, BM25Retriever 기본 개념 (06_RAG)
- `@traceable` 및 LangSmith metadata 설정 (06-Metadata-Tags-Filtering)

## 검색 품질과 LangSmith 분석

RAG 시스템에서 "답이 틀렸다"는 문제를 분석할 때, 첫 번째 의심 대상은 **검색(Retrieval)** 이에요.
LangSmith를 통해 다양한 검색 전략의 결과를 직접 비교하고, 어떤 방식이 더 좋은 문서를 가져오는지 확인할 수 있어요.

```mermaid
flowchart TD
    Q["❓ 사용자 질문"] --> R1
    Q --> R2
    Q --> R3

    subgraph R1["VectorStore Retriever"]
        R1A["similarity"] 
        R1B["mmr"]
        R1C["score_threshold"]
    end

    subgraph R2["Ensemble Retriever"]
        R2A["BM25 (키워드)"]
        R2B["FAISS (의미)"] 
        R2C["RRF 병합"]
        R2A --> R2C
        R2B --> R2C
    end

    subgraph R3["Contextual Compression"]
        R3A["Base Retriever (k=10)"]
        R3B["CrossEncoder Reranker"]
        R3C["Top-3 최종 결과"]
        R3A --> R3B --> R3C
    end

    R1 --> LS["🔍 LangSmith<br/>Trace 비교"]
    R2 --> LS
    R3 --> LS

    classDef question fill:#d4edda,stroke:#28a745,color:#155724
    classDef retriever fill:#cce5ff,stroke:#007bff,color:#004085
    classDef langsmith fill:#fff3cd,stroke:#ffc107,color:#856404

    class Q question
    class R1A,R1B,R1C,R2A,R2B,R2C,R3A,R3B,R3C retriever
    class LS langsmith
```

### 검색 방식별 특성 비교

| 검색 방식 | 특성 | LangSmith에서 보이는 것 |
|----------|------|------------------------|
| similarity | 코사인 유사도, 빠름 | 단일 Retriever Span |
| mmr | 다양성 고려, fetch_k 파라미터 | 단일 Retriever Span, 더 다양한 결과 |
| EnsembleRetriever | BM25 + FAISS 병렬 | 2개 하위 Retriever Span |
| ContextualCompression | 재순위화 | Retriever + Compressor Span |

## 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드 및 LangSmith 설정
# ---------------------------------------------------
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # FAISS + OpenMP 충돌 방지
from dotenv import load_dotenv

load_dotenv()

# 이 노트북의 Trace를 별도 프로젝트로 관리해요
os.environ["LANGSMITH_PROJECT"] = "Ch08-Retrieval-Analysis"

print(f"프로젝트: {os.environ.get('LANGSMITH_PROJECT')}")

In [ ]:
# ---------------------------------------------------
# 공통 컴포넌트 초기화
# ---------------------------------------------------
# 07 노트북에서와 동일한 VectorStore를 재구성해요
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tracers.langchain import wait_for_all_tracers

# 문서 로딩 및 분할
loader = TextLoader("data/ai_concepts.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)

# 임베딩 및 VectorStore 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents=split_docs, embedding=embeddings)

print(f"초기화 완료:")
print(f"  청크 수: {len(split_docs)}")
print(f"  VectorStore 벡터 수: {vectorstore.index.ntotal}")

## 1. search_type별 Trace 비교 (similarity vs mmr)

같은 쿼리에 대해 `similarity`와 `mmr`(Maximal Marginal Relevance) 검색 결과를 비교해요.

> 🔑 **핵심 개념**: **MMR(Maximal Marginal Relevance)**은 관련성과 다양성을 동시에 고려하는 검색 방식이에요. `lambda_mult=1.0`이면 순수 관련성 검색(similarity와 동일), `lambda_mult=0.0`이면 최대 다양성 검색이 돼요. 이 두 요소의 균형을 Trace에서 결과 비교로 확인할 수 있어요.

> 💡 **실무 팁**: 검색 결과에서 비슷한 내용의 중복 문서가 많이 나온다면 MMR을 사용해보세요. 단, MMR은 `fetch_k`개를 먼저 가져온 뒤 재정렬하므로 similarity보다 조금 느려요.

In [ ]:
# ---------------------------------------------------
# similarity vs mmr 비교 실험
# ---------------------------------------------------
# metadata 태그로 어떤 search_type인지 표시해요
# LangSmith UI에서 metadata 필터로 비교할 수 있어요

# similarity 검색 retriever
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# mmr 검색 retriever
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,             # 최종 반환 문서 수
        "fetch_k": 20,      # 초기 후보 문서 수 (fetch_k > k)
        "lambda_mult": 0.5  # 0=다양성 최대, 1=관련성 최대
    }
)

print("Retriever 설정 완료:")
print("  similarity: k=4")
print("  mmr: k=4, fetch_k=20, lambda_mult=0.5")

In [ ]:
# ---------------------------------------------------
# 같은 쿼리로 두 retriever 비교
# ---------------------------------------------------
# LangSmith에 각각 별도의 Trace가 기록돼요
# 나중에 UI에서 두 Trace를 나란히 비교할 수 있어요
from langchain_core.runnables import RunnableConfig

test_query = "머신러닝과 딥러닝의 차이점"

# metadata로 실험 설정을 태깅해요
similarity_results = similarity_retriever.invoke(
    test_query,
    config=RunnableConfig(
        metadata={"search_type": "similarity", "experiment": "ab_test_v1"}
    )
)

mmr_results = mmr_retriever.invoke(
    test_query,
    config=RunnableConfig(
        metadata={"search_type": "mmr", "experiment": "ab_test_v1"}
    )
)

# 결과 비교
print(f"쿼리: {test_query}")
print(f"\n[similarity 결과] {len(similarity_results)}개:")
for i, doc in enumerate(similarity_results):
    print(f"  {i+1}. {doc.page_content[:100]}...")

print(f"\n[MMR 결과] {len(mmr_results)}개:")
for i, doc in enumerate(mmr_results):
    print(f"  {i+1}. {doc.page_content[:100]}...")

In [ ]:
# ---------------------------------------------------
# 중복 문서 비율 확인
# ---------------------------------------------------
# similarity와 mmr 결과의 겹치는 문서 수를 확인해요
similarity_contents = set(doc.page_content for doc in similarity_results)
mmr_contents = set(doc.page_content for doc in mmr_results)

overlap = similarity_contents & mmr_contents
print("결과 비교:")
print(f"  similarity 결과 수: {len(similarity_contents)}")
print(f"  mmr 결과 수: {len(mmr_contents)}")
print(f"  겹치는 문서 수: {len(overlap)}")
print(f"  다른 문서 수: {len(similarity_contents.symmetric_difference(mmr_contents))}")

wait_for_all_tracers()
print("\n👉 LangSmith UI에서:")
print("   1. 두 Trace를 선택하여 'Compare' 버튼 클릭")
print("   2. metadata 필터: 'experiment = ab_test_v1' 으로 필터링")

## 2. EnsembleRetriever 추적

EnsembleRetriever는 BM25(키워드 검색)와 FAISS(의미론적 검색)를 병렬로 실행하여 결과를 합쳐요.
LangSmith에서 두 개의 하위 Retriever가 병렬 실행되는 구조를 확인할 수 있어요.

> 🎯 **강의 포인트**: EnsembleRetriever의 Trace를 LangSmith에서 보면 부모 Span 아래 BM25Retriever와 VectorStoreRetriever가 자식 Span으로 나타나요. 이 계층 구조를 통해 각 Retriever가 독립적으로 실행됨을 시각적으로 확인할 수 있어요.

> 💡 **실무 팁**: EnsembleRetriever의 weights는 두 검색 방식의 중요도를 조절해요. 문서가 전문 용어가 많다면 BM25 가중치를 높이고, 의미론적 질문이 많다면 FAISS 가중치를 높이세요.

In [ ]:
# ---------------------------------------------------
# EnsembleRetriever 설정: BM25 + FAISS
# ---------------------------------------------------
# BM25: 키워드 기반 통계적 검색
# FAISS: 임베딩 기반 의미론적 검색
# 두 결과를 RRF(Reciprocal Rank Fusion)로 병합해요
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# BM25 Retriever (키워드 기반)
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 4  # 반환할 문서 수

# FAISS Retriever (의미론적)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Ensemble: 두 retriever를 동일 가중치로 결합
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]  # BM25:FAISS = 50:50
)

print("EnsembleRetriever 설정 완료:")
print("  BM25 (키워드): 가중치 0.5")
print("  FAISS (의미론적): 가중치 0.5")

In [ ]:
# ---------------------------------------------------
# EnsembleRetriever 실행 및 Trace 확인
# ---------------------------------------------------
# invoke() 시 BM25와 FAISS가 병렬로 실행돼요
# LangSmith에서 두 개의 자식 Span을 볼 수 있어요
keyword_query = "LangSmith 추적 디버깅"  # 키워드 검색에 유리한 쿼리

ensemble_results = ensemble_retriever.invoke(
    keyword_query,
    config=RunnableConfig(
        metadata={
            "retriever_type": "ensemble",
            "weights": "0.5/0.5",
            "experiment": "hybrid_search_v1"
        }
    )
)

print(f"쿼리: {keyword_query}")
print(f"Ensemble 결과 ({len(ensemble_results)}개):")
for i, doc in enumerate(ensemble_results):
    print(f"\n  [{i+1}]")
    print(f"  내용: {doc.page_content[:150]}...")

wait_for_all_tracers()
print("\n👉 LangSmith UI에서 이 Trace를 클릭하면:")
print("   BM25Retriever와 VectorStoreRetriever가 병렬 자식 Span으로 보여요!")

In [ ]:
# ---------------------------------------------------
# weights 비교 실험: A/B 테스트
# ---------------------------------------------------
# 가중치 변경이 결과에 미치는 영향을 추적해요
# metadata로 각 실험 설정을 명확히 태깅해요

test_query = "자연어 처리 NLP 기술"

# 실험 A: BM25 가중치 높음
ensemble_bm25_heavy = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.7, 0.3]  # BM25 가중
)
results_a = ensemble_bm25_heavy.invoke(
    test_query,
    config=RunnableConfig(
        metadata={"experiment": "ab_weights", "variant": "A_bm25_0.7"}
    )
)

# 실험 B: FAISS 가중치 높음
ensemble_faiss_heavy = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.3, 0.7]  # FAISS 가중
)
results_b = ensemble_faiss_heavy.invoke(
    test_query,
    config=RunnableConfig(
        metadata={"experiment": "ab_weights", "variant": "B_faiss_0.7"}
    )
)

print(f"쿼리: {test_query}")
print(f"\n실험 A (BM25:FAISS = 7:3) 상위 2개:")
for doc in results_a[:2]:
    print(f"  - {doc.page_content[:100]}...")

print(f"\n실험 B (BM25:FAISS = 3:7) 상위 2개:")
for doc in results_b[:2]:
    print(f"  - {doc.page_content[:100]}...")

wait_for_all_tracers()
print("\n👉 LangSmith에서 'variant' metadata로 필터링하면")
print("   두 실험을 나란히 비교할 수 있어요!")

## 3. Reranker 적용 전후 비교

Reranker(CrossEncoder)는 1차 검색 결과를 더 정밀하게 재정렬해요.
LangSmith에서 base retriever가 10개를 가져오고 Reranker가 3개로 압축하는 과정을 볼 수 있어요.

> 🔑 **핵심 개념**: ContextualCompressionRetriever는 2단계로 작동해요: (1) base_retriever가 많은 문서를 가져오고, (2) base_compressor(Reranker)가 질문과 가장 관련성 높은 문서만 남겨요. LangSmith Trace에서 이 두 단계의 입출력을 각각 확인할 수 있어요.

> ⚠️ **자주 하는 실수**: CrossEncoder Reranker는 `sentence-transformers` 패키지와 HuggingFace 모델이 필요해요. 첫 실행 시 모델 다운로드가 발생하며 시간이 걸릴 수 있어요. 인터넷 연결이 필요해요.

**참고**: CrossEncoder 모델 설치가 필요해요:
```bash
pip install sentence-transformers
```

In [ ]:
# ---------------------------------------------------
# Reranker 없이 기본 검색 (비교 기준선)
# ---------------------------------------------------
# k=6으로 많은 후보를 가져와서 Reranker와 비교해요

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

rerank_query = "LLM 파인튜닝과 프롬프트 엔지니어링 중 어떤 것이 효과적인가요?"

base_results = base_retriever.invoke(
    rerank_query,
    config=RunnableConfig(
        metadata={
            "retriever_type": "base_faiss",
            "k": 6,
            "reranker": "none",
            "experiment": "reranker_comparison"
        }
    )
)

print(f"기본 검색 결과 ({len(base_results)}개):")
for i, doc in enumerate(base_results):
    print(f"  [{i+1}] {doc.page_content[:120]}...")

In [ ]:
# ---------------------------------------------------
# ContextualCompressionRetriever: Reranker 적용
# ---------------------------------------------------
# CrossEncoderReranker가 없는 환경을 위한 대안 구현
# 간단한 LLMChainFilter로 Reranker 역할을 시뮬레이션해요
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainFilter
from langchain_openai import ChatOpenAI

# LLMChainFilter: LLM이 각 문서의 관련성을 판단해서 필터링
# (CrossEncoderReranker보다 느리지만 추가 설치 불필요)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# LLM이 관련성 판단 (관련 없는 문서 제거)
_filter = LLMChainFilter.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter,
    base_retriever=base_retriever,  # 먼저 6개 가져온 뒤 필터링
)

print("ContextualCompressionRetriever 설정 완료:")
print("  base_retriever: k=6")
print("  compressor: LLMChainFilter (GPT-4o-mini 관련성 판단)")

In [ ]:
# ---------------------------------------------------
# Reranker 적용 후 검색 실행
# ---------------------------------------------------
# 이 Trace에서는 base_retriever + LLMChainFilter 두 단계가 보여요
compressed_results = compression_retriever.invoke(
    rerank_query,
    config=RunnableConfig(
        metadata={
            "retriever_type": "contextual_compression",
            "base_k": 6,
            "compressor": "llm_chain_filter",
            "experiment": "reranker_comparison"
        }
    )
)

print(f"쿼리: {rerank_query}")
print(f"\n기본 검색: {len(base_results)}개 → 압축 후: {len(compressed_results)}개")
print(f"\n압축 후 결과:")
for i, doc in enumerate(compressed_results):
    print(f"\n  [{i+1}] {doc.page_content[:200]}...")

wait_for_all_tracers()
print("\n👉 LangSmith에서 이 Trace를 보면:")
print("   VectorStoreRetriever (6개) → LLMChainFilter 순서로 계층 Span이 보여요")
print("   'reranker_comparison' 실험으로 기본 검색과 비교해보세요!")

## 4. LangSmith SDK로 Retriever 실행 분석

LangSmith SDK의 `client.list_runs()`를 사용하면 프로그래밍 방식으로 Retriever 실행 기록을 분석할 수 있어요.

> 💡 **실무 팁**: `list_runs()` API에는 레이트 리밋이 있어요. `start_time`을 반드시 지정하고, `select` 파라미터로 필요한 필드만 가져오세요. 7일 이내: 10 req/10s, 7일 초과: 3 req/10s.

In [ ]:
# ---------------------------------------------------
# LangSmith SDK로 Retriever 실행 기록 조회
# ---------------------------------------------------
# 방금 실행한 Retriever들의 Trace 데이터를 SDK로 분석해요
from langsmith import Client
from datetime import datetime, timedelta

client = Client()

# 최근 1시간 이내의 Retriever run만 조회
retriever_runs = list(client.list_runs(
    project_name="Ch08-Retrieval-Analysis",
    start_time=datetime.now() - timedelta(hours=1),
    run_type="retriever",  # retriever 타입만 필터링
    select=[              # 필요한 필드만 선택 (응답 크기 최소화)
        "id",
        "name",
        "inputs",
        "outputs",
        "start_time",
        "end_time",
        "extra",
    ],
))

print(f"조회된 Retriever 실행 수: {len(retriever_runs)}")

In [ ]:
# ---------------------------------------------------
# Retriever 실행 분석: 쿼리, 결과 수, 지연 시간
# ---------------------------------------------------
print("Retriever 실행 분석:")
print("-" * 80)

for run in retriever_runs[:10]:  # 최대 10개만 출력
    if run.end_time and run.start_time:
        latency = (run.end_time - run.start_time).total_seconds()
    else:
        latency = None

    # 검색 쿼리 추출
    query = "(쿼리 없음)"
    if run.inputs:
        query = run.inputs.get("query", run.inputs.get("input", str(run.inputs)[:50]))

    # 반환된 문서 수 추출
    doc_count = 0
    if run.outputs:
        docs = run.outputs.get("documents", run.outputs.get("output", []))
        if isinstance(docs, list):
            doc_count = len(docs)

    latency_str = f"{latency:.3f}s" if latency is not None else "N/A"
    print(f"  이름: {run.name}")
    print(f"  쿼리: {str(query)[:60]}")
    print(f"  문서 수: {doc_count}개, 지연: {latency_str}")
    print()

if len(retriever_runs) == 0:
    print("  (조회된 run이 없어요. Trace 제출 후 잠시 기다리거나 start_time 범위를 넓혀보세요.)")

In [ ]:
# ============================================================
# TODO: metadata 필터로 특정 실험의 run만 조회해요
# 힌트: filter 파라미터에 'eq(metadata.experiment, "reranker_comparison")'를 추가해요
# 예상 결과: reranker_comparison 실험의 run만 필터링되어 출력돼요
# ============================================================

# metadata 필터로 특정 실험 조회
experiment_runs = list(client.list_runs(
    project_name="Ch08-Retrieval-Analysis",
    start_time=datetime.now() - timedelta(hours=1),
    # metadata 필터 추가해보세요:
    # filter='eq(metadata.experiment, "reranker_comparison")',
    select=["id", "name", "extra", "start_time"],
))

print(f"실험 'reranker_comparison' run 수: {len(experiment_runs)}")
for run in experiment_runs[:5]:
    metadata = run.extra.get("metadata", {}) if run.extra else {}
    print(f"  {run.name}: {metadata}")

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **search_type 비교**: similarity와 MMR의 차이를 LangSmith Trace에서 나란히 비교할 수 있어요. metadata 태깅으로 실험을 체계적으로 관리해요
- **EnsembleRetriever Trace 구조**: BM25와 FAISS가 병렬 자식 Span으로 표시되어 각각의 검색 결과를 독립적으로 확인할 수 있어요
- **A/B 테스트 방법론**: `metadata={"experiment": "이름", "variant": "A/B"}`로 태깅하면 LangSmith UI에서 필터링과 비교가 쉬워져요
- **ContextualCompressionRetriever**: base retriever + compressor의 2단계 Trace 구조로 어떤 문서가 제거되었는지 확인할 수 있어요
- **list_runs 분석**: SDK로 retriever run만 필터링하여 쿼리별 지연 시간과 결과 수를 프로그래밍 방식으로 분석할 수 있어요

## 다음 노트북 예고

다음 `09-Full-RAG-Pipeline-Debug.ipynb`에서는 **전체 RAG 파이프라인 디버깅**을 배워요. 질문→검색→생성의 전체 흐름을 Trace로 분석하고, 병목 구간을 찾고, 문제의 원인이 검색인지 생성인지 진단하는 체계적인 방법을 익혀요.